# Homework 3

#### EE-556 Mathematics of Data - Fall 2024

In this homework we will solve constrained optimization problems.

#### Setup

<div class="alert alert-danger">
    <b>Warning</b> You will need a couple of libraries for this homework, which you can install with pip by executing the block below.
</div>

In [1]:
#!pip install -r requirements.txt

# 1. Constrained optimization - 40 points

In [2]:
from lib.part1.deblur_lib import *

#### Introduction

We have seen in the lectures that given an optimization task whose iterates live on a  convex subset $\mathcal{X} \subset \mathbb{R}^n$ we can ensure the iterates of our algorithms stay within $\mathcal{X}$ in one of two ways. 

In the first way, we use projections, computed via the proximal operator $\text{prox}_{\delta_\mathcal{X}}$. In the second way, we use linear minimization oracles  $\text{lmo}_\mathcal{X}$ within a conditional gradient framework
that take simplicial combinations of elements from the set X whereby producing iterates remaining in $\mathcal{X}$.

The following exercises will help you understand what kind of computations are involved for each of these two operators, and how their computational complexity compares. For this we will work with $\mathcal{X}$ being the set of low-rank matrices defined via the nuclear norm ball $\mathcal{X}=\lbrace X:X\in \mathbb{R}^{p\times n}, \Vert X  \Vert_* \leq \xi \rbrace$ with $\xi$ being the radius of the zero-centered nuclear norm ball.

We will first mathematically study properties of  the proximal operator $\text{prox}_{\delta_\mathcal{X}}$  (**1.1** ) and the linear minimization oracles $\text{lmo}_\mathcal{X}$ (**1.2**) onto this $\mathcal{X}$ and then compare them empirically by implementing movie recommender system (**1.3**) and image deblurring (**1.4**) algorithms.




## 1.1 Computing projections onto $\mathcal{X}$ - 10 points


#### Question 1.1.1 (2 pts)

Recall that given a set $\mathcal{X} \subset \mathbb{R}^{p \times m}$, its corresponding
    projection operator is given by
    $\mathrm{proj}_{\mathcal{X}} (\boldsymbol{Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in \mathcal{X}}\{ \|{\bf X}- \boldsymbol{Z}\|_F^2\}, \; \forall \boldsymbol{Z}\in  \mathbb{R}^{p \times m}$.
    Using the definition of the proximal operator given in class, show
    the equivalence between the projection operator and the proximal
    operator:
    $$\mathrm{proj}_{\mathcal{X}} (\boldsymbol{Z})  = \mathrm{prox}_{\delta_{\mathcal{X}}}(\boldsymbol{Z}),$$
    where $\delta_{\mathcal{X}}$ is the indicator function of
    $\mathcal{X}$ i.e.  $\delta_{\mathcal{X}}({\bf Y}) = \begin{cases} 0, \text{ if } {\bf Y}\in \mathcal{X} \\ +\infty, \text{ o.w. } \end{cases}$

<font color="blue">
The proximal operator for a function f(.) is defined as: 
    
$$\begin{equation}
\mathrm{prox}_{f}(\boldsymbol{Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}}\{ f(X) + \frac{1}{2}\|{\bf X}- \boldsymbol{Z}\|_F^2\}
\end{equation}$$

Hence, $$\mathrm{prox}_{\delta_{\mathcal{X}}}(\boldsymbol{Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in  \mathbb{R}^{p \times m}}\{ \delta_{\mathcal{X}}({\bf X}) + \frac{1}{2}\|{\bf X}- \boldsymbol{Z}\|_F^2\} = \begin{cases} \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in \mathcal{X}}\{\frac{1}{2}\|{\bf X}- \boldsymbol{Z}\|_F^2\}, \text{ if } {\bf X}\in \mathcal{X} \\ \text{undefined}, \text{ o.w. } \end{cases}$$ \
Since the constant coefficient does not affect the minimizer value, we have:

$$
\mathrm{prox}_{\delta_{\mathcal{X}}}(\boldsymbol{Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in \mathcal{X}}\{\|{\bf X}- \boldsymbol{Z}\|_F^2\} = \mathrm{proj}_{\mathcal{X}} (\boldsymbol{Z}) 
$$.

#### Question 1.1.2 (4 pts)

Moreau’s identity is a fundamental equality for $\mathrm{prox}$:
$$
 \mathrm{prox}_{f} (\textbf{x}) + \mathrm{prox}_{f^*} (\textbf{x}) = \textbf{x}, \forall \textbf{x} \in \mathbb{R}^p.
$$

Use Moreau’s identity to find the proximal operator of $\|\mathbf{x} \|_\infty$ and $\|\mathbf{x} \|_2$.

(Hint: For $f(\mathbf{x})=\|\mathbf{x}\|_p$ with $p\geq 1$, then $f^*(\mathbf{x})=\delta_{\|\mathbf{x}\|_q \leq 1}(\mathbf{x})$ where $\frac{1}{p}+\frac{1}{q}=1$.)

<font color="blue">

Q1: Find $\|\mathbf{x} \|_\infty$ 

By Moreau's identity, $\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x}) + \mathrm{prox}_{\delta_{\|\mathbf{x}\|_1 \leq 1}}(\mathbf{x}) = x$. \
Then, by using above result of proximal operator with an indicator function, $\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x}) + \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_1} \leq 1}\{\|{\bf X}- x\|_2^2\} = x$. $\quad$ ($\frac{1}{p} + \frac{1}{1} = 1$ and $p=\infty \implies q=1$)\
Hence, $\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x})
= x - \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_1} \leq 1}\{\|{\bf X}- x\|_2^2\}$. 
Now, the goal is to solve the minimization problem of $\|{\bf X}- x\|_2^2$ under the constraint of $\|X\|_1 \leq 1$. 
* $\textbf{CASE-1:}$ $\|\mathbf{x} \|_1 \leq 1$ \
It is obvious that the objective function reaches its minimum value, 0, for $\bf X^* = x$. Thus, $\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x}) = 0$
* $\textbf{CASE-2:}$ $\|\mathbf{x} \|_1 > 1$
The optimal solution for $\bf X^*$ is not obvious to find as in the previous case since $\bf X = x$ is not in the feasible set of the optimization problem. Therefore, we need to project $x$ onto the $\ell_1$-norm ball of radius 1. It is given by the soft-thresholding operation (Linehan, K., & Balan, R. (n.d.). Approximation of the Proximal Operator of the ℓ∞ Norm Using a Neural Network.):
$$ \mathbf{X}_i^* = \text{sign}(\mathbf{x}_i) \cdot \max(|\mathbf{x}_i| - \tau, 0),
\text{ where } \tau \geq 0 \, \text{is chosen such that:}
\sum_i |\mathbf{X}_i^*| = 1. \implies \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_1}\leq 1}\{\|{\bf X}- x\|_2^2\} = \mathbf{X}^*$$

A1: $$\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x}) = \begin{cases} 
x -  \mathbf{X}^*, & \text{if } \|x\|_1 > 1, \\
0, & \text{else }
\end{cases}
\quad \text{ where } \mathbf{X}^* \text{ is the projection of x onto the } \ell_1\text{-norm ball.} $$

Q2: Find $\|\mathbf{x} \|_2$ 

By Moreau's identity, $\mathrm{prox}_{\|\mathbf{x} \|_2} (\textbf{x}) + \mathrm{prox}_{\delta_{\|\mathbf{x}\|_2 \leq 1}}(\mathbf{x}) = x$. \
Then, by using above result of proximal operator with an indicator function, $\mathrm{prox}_{\|\mathbf{x} \|_2} (\textbf{x}) + \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_2} \leq 1}\{\|{\bf X}- x\|_2^2\} = x$. \
Hence, $\mathrm{prox}_{\|\mathbf{x} \|_\infty} (\textbf{x})
= x - \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_2} \leq 1}\{\|{\bf X}- x\|_2^2\}$. 
Now, the goal is to solve the minimization problem of $\|{\bf X}- x\|_2^2$ under the constraint of $\|X\|_2 \leq 1$. 
* $\textbf{CASE-1:}$ $\|\mathbf{x} \|_2 \leq 1$ \
It is obvious that the objective function reaches its minimum value, 0, for $\bf X^* = x$. Thus, $\mathrm{prox}_{\|\mathbf{x} \|_2} (\textbf{x}) = 0$
* $\textbf{CASE-2:}$ $\|\mathbf{x} \|_2 > 1$ \
By using Cauchy-Schwarz inequality ($|\langle \mathbf{u}, \mathbf{v} \rangle| \leq \|\mathbf{u}\| \|\mathbf{v}\|$), we have:
$$\|{\bf X}- x\|_2^2 \geq \frac{|\langle x,{\bf X}-x \rangle|^2}{\|x\|_2^2} \quad \text{by substituting $ \mathbf{u} = x$ and $ \mathbf{v} = {\bf X}-x$} \tag{1}$$  
By using  Cauchy-Schwarz inequality, we can also obtain this inequality,
$$|\langle x,{\bf X} \rangle| \leq \|{\bf X}\|_2 \|x\|_2 \quad \text{by substituting $ \mathbf{u} = x$ and $ \mathbf{v} = {\bf X}$} \tag{2}$$
By using given constraint,
$$|\langle x,{\bf X} \rangle| \leq \|{\bf X}\|_2 \|x\|_2 \leq  \|x\|_2  \tag{3}$$
By combining the equation (3) and (1), we can obtain the minimum value of the objective:
$$\|{\bf X}- x\|_2^2 \geq \frac{|\langle x,{\bf X}-x \rangle|^2}{\|x\|_2^2} \geq \frac{(\|x\|_2^2 - \|x\|_2)^2}{\|x\|_2^2} \tag{3}$$

The Cauchy-Schwarz equality holds if and only if $\mathbf{u}$ and $\mathbf{v}$ are in the same or opposite direction. In other words, $\mathbf{v} = \lambda\mathbf{u}$ for some scalar $\lambda$, i.e. $\bf{X}-x = \lambda_1 x$ for (1) and $\bf{X} = \lambda_2 x$ for (2) . On the other hand, we have a one more condition for equalities coming from (3), which indicates that $\|X\|_2 = 1$.
By considering all equality conditions, we can easily find that $\lambda_2 = \frac{1}{\|x\|_2}.$ Hence, ${\bf X} = \frac{x}{\|x\|_2} = \mathop{\mathrm{arg\,min}}\limits_{{\bf \|X\|_2} \leq 1}\{\|{\bf X}- x\|_2^2\}$ and  $\mathrm{prox}_{\|\mathbf{x} \|_2} (\textbf{x}) = x -  \frac{x}{\|x\|_2}.$

A2: $$\mathrm{prox}_{\|\mathbf{x} \|_2} (\textbf{x}) = \begin{cases} 
x -  \frac{x}{\|x\|_2}, & \text{if } \|x\|_2 > 1, \\
0, & \text{else }
\end{cases}$$

#### Question 1.1.3.  (4 pts)

Let ${\boldsymbol Z}= {\boldsymbol U}\boldsymbol{\Sigma} {\boldsymbol V}^\top$
    be the singular value decomposition of
    ${\boldsymbol Z}\in \mathbb{R}^{p \times m}$. Denote the diagonal of
    $\boldsymbol{\Sigma} \in \mathbb{R}^{s \times s}$ by a vector
    $\sigma \in \mathbb{R}^{s}$, where $s = \min \{ p, m \}$. Let
    $\sigma^{\ell_1}$ be the projection of $\sigma$ onto the
    $\ell_1$-norm ball
    $\{ \textbf{x}: \textbf{x}\in \mathbb{R}^{s} , \left\Vert  \textbf{x} \right\Vert_1 \leq \xi  \}$
    with radius $\xi$. Show that the projection of this matrix onto
    the nuclear norm ball
    $\mathcal{X} = \{ {\boldsymbol X}: {\boldsymbol X}\in \mathbb{R}^{p \times m} , \| {\boldsymbol X}\|_\ast \leq \xi \}$
    can be computed by projecting $\sigma$ onto the $\ell_1$ norm ball,
    i.e.,
    $$\mathrm{proj}_\mathcal{X} ({\boldsymbol Z}) = {\boldsymbol U}\boldsymbol\Sigma^{\ell_1} {\boldsymbol V}^\top,$$
    where $\Sigma^{\ell_1} \in \mathbb{R}^{s \times s}$ denotes the
    diagonal matrix with diagonal $\sigma^{\ell_1}$.

(Hint: Use Mirsky's inequality:
    $\| {\boldsymbol X}- {\boldsymbol Z}\|_F \geq \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}_{{\boldsymbol Z}}\|_F$,
    where
    $\boldsymbol{\Sigma}_{{\boldsymbol X}}, \boldsymbol{\Sigma}_{{\boldsymbol Z}} \in \mathbb{R}^{s \times s}$
    are the diagonal matrices of the singular values of
    ${\boldsymbol X}, {\boldsymbol Z}$ respectively.)

<font color="blue">

$$\| {\boldsymbol X}- {\boldsymbol Z}\|_F \geq \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F \iff \| {\boldsymbol X}- {\boldsymbol Z}\|_F^2 \geq \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2 \text{ where } {\boldsymbol X}, \boldsymbol{\Sigma}_{{\boldsymbol X}}  \in \mathcal{X} = \{ {\boldsymbol X}: {\boldsymbol X}\in \mathbb{R}^{p \times m} , \| {\boldsymbol X}\|_\ast \leq \xi \} $$
$$ \| {\boldsymbol X}- {\boldsymbol Z}\|_F^2 \geq \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2 \iff \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2 = \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 $$
$$ \text{ where  the diagonal of
    $\boldsymbol{\Sigma_x} \in \mathbb{R}^{s \times s}$ by a vector
    $\sigma_x \in \mathbb{R}^{s}$ and the diagonal of
    $\boldsymbol{\Sigma} \in \mathbb{R}^{s \times s}$ by a vector
    $\sigma \in \mathbb{R}^{s}$ }$$

$$ \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2 = \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 \implies \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 \geq \| \boldsymbol{\sigma}^{l_1} - \boldsymbol{\sigma}\|_2^2
\tag{By the definition of the projection of $\sigma$ onto  the $\ell_1$ norm ball}$$
In conclusion, we have shown that $\min \| {\boldsymbol X}- {\boldsymbol Z}\|_F^2 \geq \| \boldsymbol{\sigma}^{l_1} - \boldsymbol{\sigma}\|_2^2 $. To find the minimizer of this minimization problem, we need to look the condition where equalities hold. By definition of the projection of $\sigma$, the last equality holds when $\sigma_x = \sigma^{l_1}$, which implies that $\boldsymbol\Sigma_X = \boldsymbol\Sigma^{\ell_1} $ where $\Sigma^{\ell_1} \in \mathbb{R}^{s \times s} $ denotes the diagonal matrix with diagonal $\sigma^{\ell_1}$. Also, the equality for Mirsky's equality holds if and only if the singular vectors of ${\boldsymbol X}$ and ${\boldsymbol Z}$ are aligned in such a way that the difference between  ${\boldsymbol X}$ and ${\boldsymbol Z}$ is entirely represented by the difference in their singular values, i.e.  ${\boldsymbol X} = U\Sigma_xV^T$ while ${\boldsymbol Z} = U\Sigma V^T$. Thus,

$$ \mathrm{proj}_\mathcal{X} ({\boldsymbol Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in \mathcal{X}}\{ \|{\bf X}- \boldsymbol{Z}\|_F^2\} = \mathop{\mathrm{arg\,min}}\limits_{\substack{ \bf \Sigma_X\in \mathcal{X} \\  {\boldsymbol X} = U\Sigma_xV^T}} \{ \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2\} = \mathop{\mathrm{arg\,min}}\limits_{\substack{ \sum_{i=1}^s \sigma_{\boldsymbol X_i} \leq \xi \\  {\boldsymbol X} = U\Sigma_xV^T}} \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 = \mathop{\mathrm{arg\,min}}\limits_{\substack{ \sum_{i=1}^s  |\sigma_{\boldsymbol X_i}| \leq \xi \\  {\boldsymbol X} = U\Sigma_xV^T}} \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 = \mathrm{proj}_B(\sigma) \text{ where } B = \{\textbf{x} \in \mathbb{R}^s : \|\textbf{x}\|_1 \leq \xi\} = {\boldsymbol U}\boldsymbol\Sigma^{\ell_1} {\boldsymbol V}^\top$$


How about the projection of this matrix onto the spectral norm ball $\mathcal{S} = \{ {\boldsymbol X}: {\boldsymbol X}\in \mathbb{R}^{p \times m} , \| {\boldsymbol X}\|_2 \leq \xi \}$?

<font color="blue">

By using above observations, we can derive up to following:
$$ \mathrm{proj}_\mathcal{S} ({\boldsymbol Z}) = \mathop{\mathrm{arg\,min}}\limits_{{\bf X}\in \mathcal{S}}\{ \|{\bf X}- \boldsymbol{Z}\|_F^2\} = \mathop{\mathrm{arg\,min}}\limits_{\substack{ \bf \Sigma_X\in \mathcal{S} \\  {\boldsymbol X} = U\Sigma_xV^T}} \{ \| \boldsymbol{\Sigma}_{{\boldsymbol X}} - \boldsymbol{\Sigma}\|_F^2\} = \mathop{\mathrm{arg\,min}}\limits_{\substack{ \sigma_{\boldsymbol X_max} \leq \xi \\  {\boldsymbol X} = U\Sigma_xV^T}} \| \boldsymbol{\sigma}_{{\boldsymbol X}} - \boldsymbol{\sigma}\|_2^2 = {\boldsymbol U}\boldsymbol\Sigma_{\mathcal{S}} {\boldsymbol V}^\top $$ 

$$ \text{ where } \Sigma_{\mathcal{S}} \text{ is a diagonal matrix whose entry } d_{ii} \in \mathbb{R} = \begin{cases} 
\sigma_i, & \text{if } \sigma_i \leq \xi, \\
\xi, & \text{else }
\end{cases}
$$



## 1.2 Computing the linear minimization oracle of $\mathcal{X}$ - 2 points

Problem 1.1 shows that projection onto the nuclear norm ball requires
computing the singular value decomposition. The computational complexity
of the singular value decomposition is $\mathcal{O}(\min(m^2p,mp^2))$,
which can easily become a computational bottleneck if $m$ or $p$ are
large. This bottleneck increased the popularity of algorithms that
leverage the linear minimization oracle (lmo) instead (e.g.,
[Jaggi2013](https://proceedings.mlr.press/v28/jaggi13.html), [yurtsever2018](http://proceedings.mlr.press/v80/yurtsever18a)):
$$\text{lmo}_{\mathcal{X}}({\boldsymbol Z})  = \arg \min_{{\boldsymbol X}\in \mathcal{X}} \langle {\boldsymbol X},{\boldsymbol Z}\rangle \qquad \text{where}\qquad \langle {\boldsymbol X},{\boldsymbol Z}\rangle = \text{Tr}({\boldsymbol Z}^\top{\boldsymbol X}).$$
Note that $\text{lmo}_\mathcal{X}({\boldsymbol Z})$ is not single valued
in general. With abuse of terminology, when we say that we compute the
lmo, we actually mean that we compute an instance ${\boldsymbol X}$ such
that ${\boldsymbol X}\in \text{lmo}_\mathcal{X}({\boldsymbol Z})$.

Show that the $\text{lmo}_\mathcal{X}$ when $\mathcal{X}$ is the nuclear norm
ball:
$\mathcal{X} = \{ {\boldsymbol X}: {\boldsymbol X}\in \mathbb{R}^{p \times m} , \| {\boldsymbol X}\|_\ast \leq \xi \}$
gives the following output:
$$-\xi ~ \! \mathbf{u}\mathbf{v}^T   \in  \text{lmo}_{{\mathcal{X}}}({\boldsymbol Z}) ,$$
where $\mathbf{u}$ and $\mathbf{v}$ are the left and right singular
vectors that correspond to the largest singular value of
${\boldsymbol Z}$.

(Hint: By definition
$\xi ~ \! \mathbf{u}\mathbf{v}^T \in \mathcal{X}$. You just need to
show
$\langle {\boldsymbol X},{\boldsymbol Z}\rangle \geq \langle -\xi ~ \! \mathbf{u}\mathbf{v}^T,{\boldsymbol Z}\rangle$
for all ${\boldsymbol X}\in \mathcal{X}$.)

$$
Z = U_z \Sigma_z V_z \text{ and }X = U_x \Sigma_x V_x^T \text{ where } U_z, U_x \in \mathbb{R}^{p \times p}, V_z, V_x^T \in \mathbb{R}^{m \times m} \text{ and } \Sigma_z, \Sigma_x \in \mathbb{R}^{p \times m}
$$ 

<font color="blue">
Let decomposite X and Z as follows: 

$$
Z = U_z \Sigma_z V_z \text{ and }X = U_x \Sigma_x V_x^T \text{ where } U_z, U_x \in \mathbb{R}^{p \times p}, V_z, V_x^T \in \mathbb{R}^{m \times m} \text{ and } \Sigma_z, \Sigma_x \in \mathbb{R}^{p \times m}
$$ 

We can express X and Z as follows:
$$ Z = \sum_{i=1}^{\min \{ p, m \}}\sigma_{z_i} u_{z_i} v_{z_i}^T \text{ and }
    \tag{1}
$$
By using the equations on (1),
$$  \text{Tr}({\boldsymbol Z}^\top{\boldsymbol X}) =  \text{Tr}(( \sum_{i=1}^{\min \{ p, m \}}\sigma_{z_i} v_{z_i} u_{z_i}^T)( \sum_{j=1}^{\min \{ p, m \}}\sigma_{x_j} u_{x_j} v_{x_j}^T)) = \text{Tr}(\sum_{i=1}^{\min \{ p, m \}}\sum_{j=1}^{\min \{ p, m \}}\sigma_{z_i} v_{z_i} u_{z_i}^T\sigma_{x_j} u_{x_j} v_{x_j}^T) = \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} \text{Tr}(v_{z_i} u_{z_i}^Tu_{x_j} v_{x_j}^T) = \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} \text{Tr}(u_{z_i}^Tu_{x_j} v_{x_j}^Tv_{z_i}) $$
   $$ = \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} \text{Tr}(\langle u_{x_j}, u_{z_i} \rangle \langle v_{z_i}, v_{x_j} \rangle) = \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} \langle u_{x_j}, u_{z_i} \rangle \langle v_{z_i}, v_{x_j} \rangle \text{ (The term inside the trace becomes scalar, we can cancel out the trace.}) \tag{2}$$
   Now, we can minimize our objective by using the equation on (3):
    $$ \min_{{\boldsymbol X}\in \mathcal{X}}\langle {\boldsymbol X},{\boldsymbol Z}\rangle =  \min_{{\boldsymbol X}\in \mathcal{X}} \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} \langle u_{x_j}, u_{z_i} \rangle \langle v_{z_i}, v_{x_j} \rangle \geq - \min_{{\boldsymbol X}\in \mathcal{X}} \sum_{i=1}^{\min \{ p, m \}} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \sigma_{z_i} = -\min_{{\boldsymbol X}\in \mathcal{X}} \sum_{i=1}^{\min \{ p, m \}} \sigma_{z_i} \sum_{j=1}^{\min \{ p, m \}} \sigma_{x_j} \geq - \sum_{i=1}^{\min \{ p, m \}} \sigma_{z_i} \xi$$
This objective function attains its minimum value if and only if it satisfies these conditions:
    $$ \| {\boldsymbol X}\|_\ast =\xi, u_{x_j} = u_{z_i} \text{ and } v_{x_j} = -v_{z_i}  \text{ for } i = 1,..., \min \{ p, m \} \quad \text{ or } \quad \| {\boldsymbol X}\|_\ast =\xi, u_{x_j} = -u_{z_i} \text{ and } v_{x_j} = v_{z_i}  \text{ for } i = 1,..., \min \{ p, m \}$$

Hence, the matrix $X_{sol} = U_x \Sigma_x V_x$ where $ \Sigma_{\mathcal{x}} \text{ is a diagonal matrix whose entry } \sigma_{x_i} \in \mathbb{R} = \begin{cases} 
\xi, & \text{if } i=1, \\
0, & \text{else }
\end{cases}
$, $\quad U_x$ is an orthogonal matrix whose first column $u_{x_1} = u_{z_1}$ and $\quad V_x$ is an orthogonal matrix whose first column $v_{x_1} = -v_{z_1}$ satisfies the conditions above. By using the equation for X on (1), we have clearly shown that:
$$  X_{sol} = \sum_{i=1}^{\min \{ p, m \}}\sigma_{x_i} u_{x_i} v_{x_i}^T = -\xi u_{z_1}v_{z_1}^T \text{ $\quad$ where $ u_{z_1}$ and $v_{z_1}$ are the left and right singular
vectors that correspond to the largest singular value of
${\boldsymbol Z}$.} $$
$$\implies -\xi ~ \! \mathbf{u}\mathbf{v}^T   \in  \text{lmo}_{{\mathcal{X}}}({\boldsymbol Z}) ,$$
  

## 1.3 Comparing the scalability of $\mathrm{proj}_{\mathcal{X}} (\boldsymbol{Z})$ and $\mathrm{lmo}_{\mathcal{X}} (\boldsymbol{Z})$ - 9 points

In this exercise, we will compare the execution time of
$\mathrm{proj}_{\mathcal{X}} (\boldsymbol{Z})$ and
$\mathrm{lmo}_{\mathcal{X}} (\boldsymbol{Z})$ on two datasets provided
to you in the codes. These datasets consist of the ratings given by
MovieLens users to movies in a given list. The 100k dataset consists of
100,000 ratings from 1000 users on 1700 movies. The 1M dataset consists
of 1 million ratings from 6000 users on 4000 movies.

As you likely figured out already from the numbers above, users do not
rate all of the movies, and therefore, we model the ratings as entries
of a low-rank matrix, where rows correspond to different users and
columns correspond to different movies. A classical task in machine
learning is to predict the value of the missing entries, which is called
the matrix completion problem.

Many other tasks can be formulated as convex minimization problems,
constrained to the nuclear-norm ball, which captures a low rank model
since it is the atomic norm of rank-1 matrices (see Lecture 4). A good
optimization algorithm must ensure feasibility in a scalable way: For
instance, the famous Netflix competition data consists of 100480507
ratings that 480189 users gave to 17770 movies (much bigger than the
datasets above). Projecting a matrix of this size onto the nuclear-norm
ball is indeed demanding.

#### Question 1.3.1 (3 pts)

Implement the projection operator as a function called `proj_nuc` below. You can use the helper function `proj_L1` we define here from the `projL1.py` file.

In [3]:
from lib.part1.projL1 import projL1

Set $\xi = 3000$ and measure the computation time of the
    projection operator with the 100k and the 1M MovieLens dataset using our provided helper code, which loads the datasets, constructs the data matrix, and times the evaluation of
    the projection operator. Write the values you get in a markdown cell.
    Run and report the average timing.

In [4]:
def proj_nuc(Z, xi):
    """ This function implements the projection onto the nuclear norm ball.
    """
    
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    S_projected = projL1(S, xi)
    Z_projected = U @ np.diag(S_projected) @ Vt
    
    return Z_projected

In [ ]:
eval_completion("100k_MovieLens", proj_nuc, xi=3000)

In [ ]:
# NOTE: This one can take few minutes!
eval_completion("1M_MovieLens", proj_nuc, xi=3000)

<font color="blue">

proj for ./lib/part1/dataset/ml-100k/ub_base data takes 0.6589309215545653 sec \
proj for ./lib/part1/dataset/ml-1m/ml1m_base data takes 63.36006789207458 sec

#### Question 1.3.2 (3 pts)



The [**Newton-Schulz method**](https://www.semanticscholar.org/paper/A-STABLE-SCALING-OF-NEWTON-SCHULZ-FOR-IMPROVING-THE-Chen-Chow/3b0fecfc1781c9db401c85f2f779254e63316cdb) is an iterative algorithm for approximating the matrix inverse or inverse square root. In this question, it is used to approximate $(\boldsymbol Z^{\top} \boldsymbol Z)^{-1/2}$, which reduces the condition number of $\boldsymbol Z$ and facilitates efficient computation of its singular values.

**Formula**

For a positive definite matrix $\boldsymbol Z^{\top} \boldsymbol Z$. The Newton-Schulz iteration for approximating $(\boldsymbol Z^{\top} \boldsymbol Z)^{-1/2}$ is given by:

1. **Initialization**:
   $$
   \mathbf{X}_0 = \frac{\mathbf{I}}{\text{trace}(\boldsymbol Z^{\top} \boldsymbol Z)}
   $$
   where $\mathbf{I}$ is the identity matrix.

2. **Iteration**:
   $$
   \mathbf{X}_{t+1} = \mathbf{X}_t \cdot \left( \frac{3}{2} \mathbf{I} - \frac{1}{2} \boldsymbol Z^{\top} \boldsymbol Z \mathbf{X}_t^2 \right)
   $$

   This formula iteratively refines $\mathbf{X}_t$, converging to $(\boldsymbol Z^{\top} \boldsymbol Z)^{-1/2}$.

---

**Application in Projection**

For the nuclear-norm ball projection:
1. Newton-Schulz is used to approximate $(\boldsymbol Z^{\top} \boldsymbol Z)^{-1/2}$, transforming $\boldsymbol Z$ into a well-conditioned form:
   $$
   \mathbf{Z}_{\text{transformed}} = \mathbf{Z} \cdot \mathbf{X}_T
   $$
   where $\mathbf{X}_T $ is the result after $T$ iterations.

2. Using the transformed $\mathbf{Z}_{\text{transformed}}$, we compute the top $k$ singular values:
   $$
   \mathbf{Z}_{\text{transformed}} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^\top
   $$

3. The singular values are adjusted to satisfy the nuclear norm constraint:
   $$
   \mathbf{\Sigma}_{\text{shrink}} = \text{proj}_{L1}(\mathbf{\Sigma}, \xi)
   $$

4. The projected matrix is reconstructed as:
   $$
   \mathbf{Z}_{\text{proj}} = \mathbf{U} \mathbf{\Sigma}_{\text{shrink}} \mathbf{V}^\top
   $$

---


This combination of Newton-Schulz and partial SVD provides an effective balance between accuracy and computational scalability, especially for large-scale nuclear norm projection problems.

Implement the Newton-Schulz method with ${\mathcal{X}}$ as a function called
    `proj_newton_schulz`  below. Set again $\xi = 3000$ and measure the
    computation time for the 100k and 1M Movielens datasets. Compare these values with
    the computation time of the projection operator. You are allowed to use `scipy` (the library is already loaded).

In [5]:
def proj_newton_schulz(Z, xi, num_iter=3, k=5):
    """
    Projection onto the nuclear norm ball using Newton-Schulz.
    Args:
        Z: Input matrix.
        xi: Radius of the nuclear norm ball.
        num_iter: Newton-Schulz iterations.
        k: Number of singular values to compute.
    Returns:
        Projected matrix.
    """

    ZtZ = Z.T @ Z
    
    trace_ZtZ = np.trace(ZtZ)
    X = np.eye(ZtZ.shape[0]) / trace_ZtZ

    # Newton-Schulz iterations
    for _ in range(num_iter):
        X = X @ (1.5 * np.eye(ZtZ.shape[0]) - 0.5 * ZtZ @ X @ X)

    Z_transformed = Z @ X

    U, S, Vt = np.linalg.svd(Z_transformed, full_matrices=False)
    S_shrink = projL1(S, xi)  
    Z_projected = U[:, :k] @ np.diag(S_shrink[:k]) @ Vt[:k, :]
    
    return Z_projected

In [ ]:
eval_completion("100k_MovieLens",proj_newton_schulz,xi=3000)

In [ ]:
eval_completion("1M_MovieLens",proj_newton_schulz,xi=3000)

<font color="blue">
proj for ./lib/part1/dataset/ml-100k/ub_base data takes 0.9308882713317872 sec 
    
proj for ./lib/part1/dataset/ml-1m/ml1m_base data takes 142.0748219013214 sec


#### Question 1.3.3 (3 pts)

Implement the lmo with ${\mathcal{X}}$ as a function called
    `lmo_nuc`  below. Set again $\xi = 3000$ and measure the
    computation time for the 100k and 1M Movielens datasets. Compare these values with
    the computation time of the projection operator. You are allowed to use `scipy` (the library is already loaded).

In [6]:
def lmo_nuc(Z, xi):
    """ This function implements the lmo operator for the nuclear norm ball constraint.
    """

    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    
    u1 = U[:, 0]  # First left singular vector
    v1 = Vt[0, :]  # First right singular vector
    
    X_lmo = -xi * np.outer(u1, v1)

In [ ]:
eval_completion("100k_MovieLens",lmo_nuc,xi=3000)

In [ ]:
# NOTE: This one can take few minutes!
eval_completion("1M_MovieLens",lmo_nuc,xi=3000)

<font color=‘blue’>
proj for ./lib/part1/dataset/ml-100k/ub_base data takes 0.5326841831207275 sec

proj for ./lib/part1/dataset/ml-1m/ml1m_base data takes 66.47396063804626 sec


## 1.4 Frank-Wolfe for blind image deblurring - 19 points

You are working with the local police to help identify a license plate
of a car involved in a crime scene investigation. Unfortunately, the
CCTV image of the car is blurry. In this exercise, we simulate this
scenario with a deblurred license plate image found from the
internet.

Deblurring is an instance of the blind deconvolution problem: Given two
unknown vectors $\textbf{x},  {\boldsymbol w}\in \mathbb{R}^L$, we
observe their circular convolution
$\textbf{y}=  {\boldsymbol w}*\textbf{x}$, i.e.,
$$
y_\ell = \sum_{\ell'=1}^L w_{\ell'} x_{\ell - \ell' + 1},
$$ 
where the index $\ell - \ell' + 1$ in the sum is understood to be modulo $L$.

Blind deconvolution seeks to separate ${\boldsymbol w}$ and
$\textbf{x}$, given $\textbf{y}$. The operative word *blind* comes from
the fact that we do not have much prior information about the signals.
In this case, what we can assume is that ${\boldsymbol w}$ and
$\textbf{x}$ belong to *known* subspaces of $\mathbb{R}^L$ of dimension
$K$ and $N$, i.e., we write $$\begin{aligned}
{\boldsymbol w}&= {\boldsymbol B}{\boldsymbol h}, \quad {\boldsymbol h}\in \mathbb{R}^K \\
\textbf{x}&= {\boldsymbol C}{\boldsymbol m}, \quad {\boldsymbol m}\in \mathbb{R}^N
\end{aligned}$$ for some $L \times K$ matrix ${\boldsymbol B}$ and
$L \times N$ matrix ${\boldsymbol C}$. The columns of these matrices
form bases for the subspaces in which ${\boldsymbol w}$ and $\textbf{x}$
live.

As we have seen in Homework 1, natural images have sparse wavelet
expansions. Hence, the image $\textbf{x}$ can be expressed as
$\textbf{x}= {\boldsymbol C}{\boldsymbol m}$ with ${\boldsymbol C}$ is
the matrix formed by a subset of the columns of the wavelet transform
matrix. In addition, the blur kernel ${\boldsymbol w}$ is typically due
to simple or "sparse" motion, which can be written as
${\boldsymbol w}= {\boldsymbol B}{\boldsymbol h}$ with ${\boldsymbol B}$
is the matrix formed by a subset of the columns of the identity matrix.

In deblurring, $\textbf{x}$ corresponds to the image we want to recover
(i.e., the license plate) and ${\boldsymbol w}$ to a 2D blur kernel.
Thus, the 2D convolution $\textbf{y}=  {\boldsymbol w}*\textbf{x}$
produces a blurred image. We assume that we know or can estimate the
support of the blur kernel (i.e., the location of its nonzero elements).
In real applications, the support can be estimated by an expert using
the physical information such as the distance of object to the focus and
the camera, the speed of the camera and/or the object, camera shutter
speed, etc.

In this experiment, we use a very rough estimate for the support - a box
at the center of the domain, whose size we have roughly tuned.
Interestingly, it is possible to make the plate readable even in this
setting.

In [ ]:
from lib.part1.deblur_lib import setup_show
setup_show()

###  Reformulating the problem

We now reformulate the blind image deconvolution problem, so that we can
apply the constrained optimization algorithms we have seen in the
course. Let ${\boldsymbol b}$ be the $L$-point normalized discrete
Fourier transform (DFT) of the observation $\textbf{y}$, i.e,
${\boldsymbol b}= \mathbf{F} \textbf{y}$, where $F$ is the DFT matrix.
Then, ${\boldsymbol b}$ can be written as
${\boldsymbol b}= {\boldsymbol A}({\bf X})$ where
${\bf X}= {\boldsymbol h}{\boldsymbol m}^\top$ and ${\boldsymbol A}$ is
a linear operator. Explicit expression of this linear operator
${\boldsymbol A}$ is out of the scope of this homework, c.f.,
[Ahmed2014](https://ieeexplore.ieee.org/document/6680763/) for further details. This reformulation allows us to
express $\textbf{y}$, which is a nonlinear combination of the
coefficients of ${\boldsymbol h}$ and ${\boldsymbol m}$, as a linear
combination of the entries of their outer product
${\bf X}= {\boldsymbol h}{\boldsymbol m}^\top$. Note that given
${\boldsymbol B}$ and ${\boldsymbol C}$, recovering ${\boldsymbol m}$
and ${\boldsymbol h}$ from ${\boldsymbol b}$ is the same as recovering
$\textbf{x}$ and ${\boldsymbol w}$ from $\textbf{y}$.

Since ${\bf X}$ is a rank one matrix, we can use the nuclear norm to
enforce approximately low-rank solutions. Then, we can formulate the
blind deconvolution problem as follows:
$$
\begin{align}
{\boldsymbol X}^\star \in \arg \min_{ {\boldsymbol X}} \bigg\{ \frac{1}{2} \| \mathbf{A}({\boldsymbol X}) - {\boldsymbol b}\|_2^2 :  \| {\boldsymbol X}\|_\ast \leq \xi, ~{\boldsymbol X}\in \mathbb{R}^{p\times m}   \bigg\}, \tag{4}
\end{align}
$$
where $\xi > 0$ is a tuning parameter.

Note that our problem is constrained to the nuclear norm ball
$\mathcal{X} = \{ {\boldsymbol X}: {\boldsymbol X}\in \mathbb{R}^{p \times m} , \| {\boldsymbol X}\|_\ast \leq \xi \}$.

We will apply the Frank-Wolfe algorithm to solve the optimization
problem given in \eqref{eq:FWform}. The Frank-Wolfe algorithm is one of the earliest
algorithms that avoids projections. Instead of projections, it leverages
lmos (for a very good survey see [Jaggi2013](https://proceedings.mlr.press/v28/jaggi13.html)):
$$\mathrm{lmo}(\nabla f ({\boldsymbol Z})) = \arg \min_{{\boldsymbol X}\in \mathcal{X}} ~ \langle \nabla f ({\boldsymbol Z}), {\boldsymbol X}\rangle,$$
where
$\mathcal{X} = \{ {\boldsymbol X}: \| {\boldsymbol X}\|_\ast \leq \xi, ~{\boldsymbol X}\in \mathbb{R}^{p\times m} \}$
as in Part 1. It applies to the generic constrained minimization
template with a smooth objective function,
$\min_{\boldsymbol X}\{ f({\boldsymbol X}) : {\boldsymbol X}\in \mathcal{X}, \, \mathcal{X} \text{ - convex, compact}  \}$
as follows:

----


### Frank-Wolfe's algorithm

1. Choose ${\boldsymbol X}^0 \in\mathcal{X}$.

2. For $k=0, 1, \ldots$ perform:
$$
\begin{cases}
  \hat{{\bf X}}^k &:= \mathrm{lmo}(\nabla f ({\boldsymbol X}^k)), \\
  {\bf X}^{k+1} &:= (1-\eta_k){\bf X}^k + \eta_k\hat{{\bf X}}^k,
\end{cases}
$$ 
where $\eta_k := {2}/{(k+2)}$

---



#### Question 1.4.1 (4 pts)

Recall that the Frank-Wolfe algorithm applies only for
    smooth objectives. Show that the objective function is smooth in the sense its gradient is Lipschitz continuous.

<font color="blue">
    
To show that the objective function is smooth, it is needed to prove that its gradient is Lipschitz continuous. Specifically, we must demonstrate the existence of a constant 
$L$ such that:
$$ 
\| \nabla f(X_1) - \nabla f(X_2) \|_F \leq L \| X_1 - X_2 \|_F, \quad \text{for all } X_1, X_2 \in \mathbb{R}^{p \times m}, \tag{1}
$$
$$ \text{ where} \|\cdot\|_F \text{ denotes the Frobenius norm. }$$
Let take the gradient of out objective function: $f(X) = \frac{1}{2} \| \mathbf{A}({\boldsymbol X}) - {\boldsymbol b}\|_2^2$ 
$$ \nabla f(X) = \mathbf{A}^T(\mathbf{A}(X) - b) \quad \text{ where } \mathbf{A}^T \text{ is the adjoint operator of } A. \tag{2}$$
By substituting the equation (2) into equation (1), we obtain the following:
$$ \| \nabla f(X_1) - \nabla f(X_2) \|_F = \| \mathbf{A}^T(\mathbf{A}(X_1) - \mathbf{A}(X_2)) \|_F$$
$$  \| \mathbf{A}^T(\mathbf{A}(X_1) - \mathbf{A}(X_2)) \|_F \leq \| \mathbf{A}^T \|  \|\mathbf{A}(X_1) - \mathbf{A}(X_2) \|_F \tag{By using the submultiplicative property of norms}$$
$$ \| \mathbf{A}^T \|  \|\mathbf{A}(X_1) - \mathbf{A}(X_2) \|_F = \| \mathbf{A}^T \|  \|\mathbf{A}(X_1 - X_2) \|_F \tag{By using the linearity of $\mathbf{A}$ }$$
$$  \| \mathbf{A}^T \|  \|\mathbf{A}(X_1 - X_2) \|_F \leq  \| \mathbf{A}^T \| \| \mathbf{A} \| \|(X_1 - X_2) \|_F \tag{By using the submultiplicative property of norms}$$

As a result, we have: 
$$ \| \nabla f(X_1) - \nabla f(X_2) \|_F  \leq  \| \mathbf{A}^T \| \| \mathbf{A} \| \|(X_1 - X_2) \|_F \tag{3}$$
From the equation (3), we can clearly show that the objective function is smooth in the sense its gradient is Lipschitz continuous:
$$ \| \nabla f(X_1) - \nabla f(X_2) \|_F  \leq  L \|(X_1 - X_2) \|_F  \quad \text{ where } L = \| \mathbf{A}^T \| \| \mathbf{A} \| .$$  

### Implementation of Frank-Wolfe

Complete the missing lines of the Frank-Wolfe update below.
We provide you the linear operators that you need to compute the LMO in the code. Note that we do not need to
    store and use the linear operator ${\boldsymbol A}$ in the ambient
    dimensions. In fact, for storage and arithmetic efficiency, we
    should avoid explicitly writing ${\boldsymbol A}$. You can find more
    details about this aspect as comments in the code.

In [ ]:
from lib.part1.deblur_lib import *

We will track three states:

- `X`: corresponding to $\boldsymbol X$
- `AX`: corresponding to $\boldsymbol{AX}$
- `k`: the iterate count

In [ ]:
def init_state():
    return OptState(X=0.0, AX=0.0, k=1)

#### Question 1.4.2 (2 points) Complete the gradient computation of $f(\boldsymbol X)$
You have access to the following operators and variables:
    
- `A_T(Z)`: Computes $\boldsymbol A^\top \boldsymbol Z$ for some $\boldsymbol Z$.
- `Aoper`: Computes $\boldsymbol A \boldsymbol X$ for 1-rank matrix $\boldsymbol X$ using its SVD decomposition.
- `b`: corresponds to $\boldsymbol b$

**Remark**: The `grad` method will receive `AX` and not `X` for computational efficiency reasons.

In [ ]:
f = Function(
    f = lambda AX: 0.5*np.linalg.norm(AX - b, 2)**2,
    grad = lambda AX: A_T(AX - b),
)

#### Question 1.4.3 (5 points) Complete the LMO

You are allowed to use `scipy`.

In [ ]:
def lmo(Grad, xi=1000):
    """ This function implements the lmo operator.
    """
    u, S, v = svds(Grad, k=1, which='LM')
    Xhat = -xi * np.outer(u, v)
    AXhat = Aoper(u, -xi, v.T)
    
    return (Xhat, AXhat)

#### Question 1.4.4 (5 points) Complete the Frank-Wolfe update

Fill in the missing update of `X` using the LMO from the previous exercise.

In [ ]:
def state_update(f, state):
    X, AX, k = state
    
    Grad = f.grad(AX)  # Gradient of f
    Xhat, AXhat = lmo(Grad, xi=100)  
    
    # Step size
    eta_k = 2 / (k + 2)
    
    # Update X
    next_X = (1.0 - eta_k) * X + eta_k * Xhat

    # Update AX
    next_AX = (1.0 - eta_k) * AX + eta_k * AXhat

    return OptState(next_X, next_AX, k+1)
    

In [ ]:
opt_algorithm = OptAlgorithm("FrankWolfe", init_state, state_update)

#### Question 1.4.5 (3 points) Run Frank-Wolfe

Tune the 
    parameter $\xi$ until the license plate number becomes readable.
    What is the license plate number? 


In [ ]:
# Run Frank-Wolfe's method
xFW = run_frank_wolfe(f, opt_algorithm)

<font color="blue">
    
We can observe that the plate number is J209 LTL.